# Assignment 1 - Part 2 — Backward Pass
**Introduction to Deep Learning, THWS**

In Part 1 you built the forward pass of a neural network from scratch.
Now we implement the backward pass — the mechanism that makes learning possible.

We follow the local/global gradient logic from the lecture:
each node in the computation graph first computes its **local gradient** (how its output changes with its input, independent of the rest of the graph), and then the **global gradient** is obtained by multiplying the local gradient by the upstream gradient via the chain rule.

**How to work:**
- Copy the Part 2 (A1P2) files into the same folder as the Part 1 files (A1P1)
- Backward functions go in `functions.py`.
- Backward methods for the classes go in `layers_backward.py`.
- Edit files in your editor, save, then re-run cells here to test.
- Use only PyTorch tensor operations — no `torch.nn`, no `numpy`.

**Submission:** zip the complete `A2/` folder and submit via Moodle before the deadline.

In [74]:
%load_ext autoreload
%autoreload 2

import torch
from helpers import load_data, numerical_gradient, grad_checker

X, y = load_data()
# X = X.double()
# y = y.double()
print(f'Data loaded: X={X.shape}, y={y.shape}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Data loaded: X=torch.Size([500, 4]), y=torch.Size([500, 1])


---
## Section 1 — MSE backward

MSE is the **root** of our computation graph — there is no upstream gradient.
This means the local and global gradients are always identical.
We therefore have a single backward function for MSE.

**Task 1.1** Derive the gradient of the scalar MSE loss $L = (\hat{y} - y)^2$ w.r.t. $\hat{y}$ on paper, then implement `mse_backward_scalar` in `functions.py`.

In [75]:
from functions import mse_backward_scalar

x0     = X[0, 0].item()
y0     = y[0].item()
theta  = torch.tensor([0.0, 1.0])
y_pred = theta[1] * x0 + theta[0]

g = mse_backward_scalar(y_pred, y0)
print ("X =", x0, y0)
print(f'y_pred: {y_pred:.4f},  y: {y0:.4f}')
print(f'dL/d(y_pred): {g:.4f}')

X = 0.5304189920425415 -0.9118689894676208
y_pred: 0.5304,  y: -0.9119
dL/d(y_pred): 2.8846


In [76]:
# verify with numerical gradient
f_scalar = lambda yp: torch.tensor((yp.item() - y0) ** 2)
g_num    = numerical_gradient(f_scalar, torch.tensor([y_pred]))
grad_checker(torch.tensor([g]), g_num, name='mse_backward_scalar')

[mse_backward_scalar] max absolute error: 2.89e-04
[mse_backward_scalar] max relative error: 1.00e-04
[mse_backward_scalar] Gradient check PASSED.


**Question 1.1** What does the sign of the gradient tell you about whether a prediction is too high or too low?

*Answer:* When the gradient is positive, it means our prediction is higher than the actual value.
For example if ŷ = 7.0 and y = 5.0, then gradient = 2 × (7.0 - 5.0) = +4.0, which tells us we predicted too high and need to bring the prediction down.
When the gradient is negative, it means our prediction is lower than the actual value. 
For example if ŷ = 3.0 and y = 5.0, then gradient = 2 × (3.0 - 5.0) = -4.0, which tells us we predicted too low and need to bring the prediction up.

Now extend to the batch case: $L = \frac{1}{N} \sum_{i=1}^N (\hat{y}^{(i)} - y^{(i)})^2$

**Task 1.2** Derive the gradient w.r.t. each $\hat{y}^{(i)}$ on paper, then implement `mse_backward` in `functions.py`.

In [77]:
from functions import mse_backward
from layers import linear_forward

torch.manual_seed(0)
theta_1 = torch.randn(1, 4)
theta_0 = torch.zeros(1, 1)
y_pred  = linear_forward(X, theta_1, theta_0)

g = mse_backward(y_pred, y)
print(f'gradient shape:  {g.shape}')
print(f'first 5 values:  {g[:5].squeeze().tolist()}')

gradient shape:  torch.Size([500, 1])
first 5 values:  [0.0016534074675291777, 0.0017999461852014065, 0.005688684526830912, -0.0037710745818912983, -0.004854164551943541]


In [78]:
# verify with numerical gradient on a small batch
from layers import mse_forward

X_small, y_small = X[:5], y[:5]
torch.manual_seed(0)
yp_small = linear_forward(X_small, theta_1, theta_0)
g_small  = mse_backward(yp_small, y_small)
g_num    = numerical_gradient(lambda yp: mse_forward(yp, y_small), yp_small)
grad_checker(g_small, g_num, name='mse_backward')

[mse_backward] max absolute error: 4.06e-04
[mse_backward] max relative error: 2.17e-03
[mse_backward] Gradient check FAILED — check your implementation.


In [80]:
# #  NEW SECTION - same but with double precision
# # In this tiny errors stay tiny
# # relative error ~1e-10 < threshold 1e-3
# from layers import mse_forward

# X_small, y_small = X[:5].double(), y[:5].double()   # ← convert to double
# torch.manual_seed(0)
# yp_small = linear_forward(X_small, theta_1.double(), theta_0.double())
# g_small  = mse_backward(yp_small, y_small)
# g_num    = numerical_gradient(lambda yp: mse_forward(yp, y_small), yp_small)
# grad_checker(g_small, g_num, name='mse_backward')

---
## Section 2 — Linear backward

Consider the linear layer: $Z = X \Theta_1^\top + \Theta_0$

We split the backward pass into two steps:
1. **Local gradients** (`_lgrad`) — partial derivatives of $Z$ w.r.t. its inputs, independent of the rest of the graph
2. **Global gradients** (`_ggrad`) — apply the chain rule with upstream $G_{out}$. In the scalar case this is elementwise multiplication. In the matrix case the chain rule produces matrix products — think carefully about how the dimensions need to work out.

**Task 2.1** Start with the scalar case $z = \theta_1 x + \theta_0$.
Derive the three local gradients $\partial z / \partial \theta_0$, $\partial z / \partial \theta_1$, and $\partial z / \partial x$ on paper.

**Question 2.1** What are the three local gradients?

*Answer:* The three local gradients for z = θ₁x + θ₀ are:

dz/dθ₀ = 1 because θ₀ is added directly so changing it by 1 always changes z by exactly 1,   
dz/dθ₁ = x because θ₁ is multiplied by x so its effect depends on the input value,  
dz/dx = θ₁ because x is multiplied by θ₁ so its effect depends on the weight value

**Task 2.2** Implement `linear_lgrad_scalar` and `linear_ggrad_scalar` in `functions.py`.

> **Hint:** When returning parts of existing tensors as local gradients, use `.clone()` to avoid unexpected aliasing — otherwise modifying the returned value could affect the original tensor.

In [81]:
from functions import linear_lgrad_scalar, linear_ggrad_scalar

x     = X[0, 0]
theta = torch.tensor([0.0, 1.0])   # (theta_0, theta_1)
gout  = torch.tensor(1.0)          # upstream gradient — start with 1

# step 1: local gradients
lgrad_theta_0, lgrad_theta_1, lgrad_x = linear_lgrad_scalar(x, theta)
print('Local gradients:')
print(f'  dz/d(theta_0) = {lgrad_theta_0:.4f}')
print(f'  dz/d(theta_1) = {lgrad_theta_1:.4f}')
print(f'  dz/dx         = {lgrad_x:.4f}')

# step 2: global gradients — with gout=1, global = local
ggrad_theta_0, ggrad_theta_1, ggrad_x = linear_ggrad_scalar(gout, x, theta)
theta.g = torch.stack([ggrad_theta_0, ggrad_theta_1])
x.g     = ggrad_x
print('\nGlobal gradients (gout=1):')
print(f'  theta.g = {theta.g}')
print(f'  x.g     = {x.g:.4f}')

Local gradients:
  dz/d(theta_0) = 1.0000
  dz/d(theta_1) = 0.5304
  dz/dx         = 1.0000

Global gradients (gout=1):
  theta.g = tensor([1.0000, 0.5304])
  x.g     = 1.0000


In [82]:
# now use actual upstream gradient from mse_backward_scalar
y0     = y[0].item()
y_pred = (theta[1] * x + theta[0]).item()
gout   = torch.tensor(mse_backward_scalar(y_pred, y0))

ggrad_theta_0, ggrad_theta_1, ggrad_x = linear_ggrad_scalar(gout, x, theta)
theta.g = torch.stack([ggrad_theta_0, ggrad_theta_1])
x.g     = ggrad_x

print(f'gout: {gout.item():.4f}')
print(f'theta.g: {theta.g}')
print(f'x.g:     {x.g:.4f}')

f_loss = lambda t: torch.tensor((t[1].item() * x.item() + t[0].item() - y0) ** 2)
grad_checker(theta.g, numerical_gradient(f_loss, theta), name='linear scalar')

gout: 2.8846
theta.g: tensor([2.8846, 1.5300])
x.g:     2.8846
[linear scalar] max absolute error: 5.79e-04
[linear scalar] max relative error: 3.78e-04
[linear scalar] Gradient check PASSED.


**Task 2.3** Now extend to the batch case $Z = X \Theta_1^\top + \Theta_0$.

**Question 2.2** What are the shapes of $G_{out}$, $\partial L / \partial \Theta_1$, $\partial L / \partial \Theta_0$, and $\partial L / \partial X$? Use the shape information in the docstrings of `linear_lgrad` and `linear_ggrad`.

*Answer:* When I looked at the docstrings, the shapes are:

G_out shape = (N, out_features) = (500, 1) one upstream gradient value for each sample and each output neuron  
dL/dΘ₁ shape = (out_features, in_features) = (1, 4) exactly same shape as weight matrix because every weight gets its own gradient  
dL/dΘ₀ shape = (1, out_features) = (1, 1) exactly same shape as bias vector because every bias gets its own gradient  
dL/dX shape = (N, in_features) = (500, 4) exactly same shape as input X because every input feature of every sample gets its own gradient

**Question 2.3** Why do we need $\partial L / \partial X$ in addition to the parameter gradients?

*Answer:* We need dL/dX because our network has multiple layers and the gradient has to travel backwards through all of them. When I think about it, dL/dX of one layer becomes the upstream gradient for the layer before it. So without dL/dX the gradient would get stuck and earlier layers would never receive any information about how to update their weights. Basically it is not for updating the input data. It is just a messenger that keeps the chain rule moving backwards through the whole network.

**Task 2.4** Implement `linear_lgrad` and `linear_ggrad` in `functions.py`.

In [83]:
from functions import linear_lgrad, linear_ggrad
from layers import linear_forward

torch.manual_seed(0)
theta_1 = torch.randn(1, 4)
theta_0 = torch.zeros(1, 1)
y_pred  = linear_forward(X, theta_1, theta_0)
gout    = mse_backward(y_pred, y)

# step 1: local gradients
lgrad_t1, lgrad_t0, lgrad_ins = linear_lgrad(X, theta_1, theta_0)
print('Local gradient factors:')
print(f'  lgrad_theta_1 factor: {lgrad_t1.shape}')
print(f'  lgrad_theta_0 factor: {lgrad_t0.shape}')
print(f'  lgrad_ins:            {lgrad_ins.shape}')

# step 2: global gradients
theta_1.g, theta_0.g, X.g = linear_ggrad(gout, X, theta_1, theta_0)
print('\nGlobal gradients:')
print(f'  theta_1.g: {theta_1.g.shape}')
print(f'  theta_0.g: {theta_0.g.shape}')
print(f'  X.g:       {X.g.shape}')

Local gradient factors:
  lgrad_theta_1 factor: torch.Size([500, 4])
  lgrad_theta_0 factor: torch.Size([500, 1])
  lgrad_ins:            torch.Size([1, 4])

Global gradients:
  theta_1.g: torch.Size([1, 4])
  theta_0.g: torch.Size([1, 1])
  X.g:       torch.Size([500, 4])


In [84]:
# verify
from layers import mse_forward

f_t1 = lambda t1: mse_forward(linear_forward(X, t1, theta_0), y)
f_t0 = lambda t0: mse_forward(linear_forward(X, theta_1, t0), y)
f_x  = lambda x:  mse_forward(linear_forward(x, theta_1, theta_0), y)

grad_checker(theta_1.g, numerical_gradient(f_t1, theta_1), name='theta_1')
grad_checker(theta_0.g, numerical_gradient(f_t0, theta_0), name='theta_0')
grad_checker(X.g,       numerical_gradient(f_x, X),        name='X')

[theta_1] max absolute error: 5.61e-03
[theta_1] max relative error: 4.89e-03
[theta_1] Gradient check FAILED — check your implementation.
[theta_0] max absolute error: 1.27e-07
[theta_0] max relative error: 1.27e+01
[theta_0] Gradient check FAILED — check your implementation.
[X] max absolute error: 5.54e-03
[X] max relative error: 4.67e+05
[X] Gradient check FAILED — check your implementation.


In [57]:
# # NOTE: The original verify cell fails due to float32 precision limitations.
# # float32 (~7 digits) is not accurate enough for numerical gradient checking
# # with h=1e-4. Converting to float64 (double, ~15 digits) fixes this.
# # No changes to the gradient implementation — only precision is changed.
# # verify
# from layers import mse_forward

# # ← convert everything to double here too!
# theta_1_d = theta_1.double()
# theta_0_d = theta_0.double()
# X_d = X.double()
# y_d = y.double()

# # recompute gradients in double
# theta_1_d.g, theta_0_d.g, X_d.g = linear_ggrad(
#     mse_backward(linear_forward(X_d, theta_1_d, theta_0_d), y_d),
#     X_d, theta_1_d, theta_0_d
# )

# f_t1 = lambda t1: mse_forward(linear_forward(X_d, t1, theta_0_d), y_d)
# f_t0 = lambda t0: mse_forward(linear_forward(X_d, theta_1_d, t0), y_d)
# f_x  = lambda x:  mse_forward(linear_forward(x, theta_1_d, theta_0_d), y_d)

# grad_checker(theta_1_d.g, numerical_gradient(f_t1, theta_1_d), name='theta_1')
# grad_checker(theta_0_d.g, numerical_gradient(f_t0, theta_0_d), name='theta_0')
# grad_checker(X_d.g,       numerical_gradient(f_x, X_d),        name='X')

---
## Section 3 — ReLU backward

Recall the ReLU: $a = \text{relu}(z) = \max(0, z)$

**Task 3.1** Derive the local gradient $\partial a / \partial z$ on paper. Consider three cases: $z > 0$, $z < 0$, and $z = 0$.

**Question 3.1** What is the local gradient for each case?

*Answer:* ...

**Task 3.2** Implement `relu_lgrad_scalar`, `relu_ggrad_scalar`, `relu_lgrad`, and `relu_ggrad` in `functions.py`.

In [85]:
from functions import relu_lgrad_scalar, relu_ggrad_scalar

for val in [2.0, -1.5, 0.0]:
    z    = torch.tensor(val)
    gout = torch.tensor(1.0)

    # step 1: local gradient
    lgrad = relu_lgrad_scalar(z)

    # step 2: global gradient
    z.g = relu_ggrad_scalar(gout, z)

    print(f'z={val:5.1f}  lgrad={lgrad:.1f}  z.g={z.g:.1f}')

z=  2.0  lgrad=1.0  z.g=1.0
z= -1.5  lgrad=0.0  z.g=0.0
z=  0.0  lgrad=0.0  z.g=0.0


**Question 3.2** What happens to the gradient flowing through an inactive unit (pre-activation $< 0$)? What does this mean for the parameters upstream of that unit?

*Answer:* When a unit is inactive meaning its pre-activation is less than 0, the gradient that flows through it becomes zero because ReLU blocked that value during forward pass. So when we multiply the upstream gradient by the local gradient (which is 0), the result is always 0 no matter what.
This means the parameters that are upstream of that inactive unit get zero gradient and their weights do not update at all in that training step. Basically they learn nothing from that sample.
From what I observed, if ReLU killed a value during forward pass, the backward pass also kills the gradient at that same position so the layers before it get no information about how to improve.

---
## Section 4 — Backward pass through the classes

Open `layers_backward.py`. It contains commented-out class skeletons.
For each class:
1. Copy your `__init__` and `forward` implementation from Part 1's `layers.py`
2. Add the `backward` method

Each `backward` method should call the corresponding `_ggrad` function from `functions.py`, assign the returned values to `.g` attributes on the tensors, and return `self.ins.g` so the gradient can propagate to the preceding layer.

**Task 4.1** Implement `Linear.backward`.

In [86]:
from layers_backward import Linear, ReLU, Model

torch.manual_seed(0)
layer = Linear(theta_1=torch.randn(1, 4), theta_0=torch.zeros(1, 1))
y_pred = layer.forward(X)
gout   = mse_backward(y_pred, y)

layer.backward(gout)

print(f'layer.theta_1.g shape: {layer.theta_1.g.shape}')   # (1, 4)
print(f'layer.theta_0.g shape: {layer.theta_0.g.shape}')   # (1, 1)
print(f'layer.ins.g shape:     {layer.ins.g.shape}')        # (500, 4)

layer.theta_1.g shape: torch.Size([1, 4])
layer.theta_0.g shape: torch.Size([1, 1])
layer.ins.g shape:     torch.Size([500, 4])


In [87]:
from layers_backward import Linear, ReLU, Model
# verify against numerical gradient
torch.manual_seed(0)
layer = Linear(theta_1=torch.randn(1, 4), theta_0=torch.zeros(1, 1))
layer.forward(X)
layer.backward(mse_backward(layer.outs, y))

f_t1 = lambda t1: mse_forward(linear_forward(X, t1, layer.theta_0), y)
f_t0 = lambda t0: mse_forward(linear_forward(X, layer.theta_1, t0), y)

grad_checker(layer.theta_1.g, numerical_gradient(f_t1, layer.theta_1), name='Linear.theta_1')
grad_checker(layer.theta_0.g, numerical_gradient(f_t0, layer.theta_0), name='Linear.theta_0')

[Linear.theta_1] max absolute error: 5.61e-03
[Linear.theta_1] max relative error: 4.89e-03
[Linear.theta_1] Gradient check FAILED — check your implementation.
[Linear.theta_0] max absolute error: 1.27e-07
[Linear.theta_0] max relative error: 1.27e+01
[Linear.theta_0] Gradient check FAILED — check your implementation.


In [53]:
# from layers_backward import Linear, ReLU, Model
# # verify against numerical gradient
# torch.manual_seed(0)
# layer = Linear(theta_1=torch.randn(1, 4).double(), 
#                theta_0=torch.zeros(1, 1).double())  # ← .double()

# layer.forward(X.double())                            # ← .double()
# layer.backward(mse_backward(layer.outs, y.double())) # ← .double()

# f_t1 = lambda t1: mse_forward(linear_forward(X.double(), t1, layer.theta_0), y.double())
# f_t0 = lambda t0: mse_forward(linear_forward(X.double(), layer.theta_1, t0), y.double())

# grad_checker(layer.theta_1.g, numerical_gradient(f_t1, layer.theta_1), name='Linear.theta_1')
# grad_checker(layer.theta_0.g, numerical_gradient(f_t0, layer.theta_0), name='Linear.theta_0')

**Task 4.2** Implement `ReLU.backward` in `layers_backward.py`.

In [88]:
relu = ReLU()
z    = torch.tensor([[-1.5,  0.5,  2.0],
                     [ 1.0, -0.3,  0.0]])
a    = relu.forward(z)
gout = torch.ones_like(a)   # upstream gradient all ones

relu.backward(gout)
print(f'z:\n{z}')
print(f'z.g:\n{relu.ins.g}')

z:
tensor([[-1.5000,  0.5000,  2.0000],
        [ 1.0000, -0.3000,  0.0000]])
z.g:
tensor([[0., 1., 1.],
        [1., 0., 0.]])


**Task 4.3** Implement `Model.backward` in `layers_backward.py`.

**Question 4.1** Why must the backward pass run through layers in reverse order?

*Answer:* The backward pass must run in reverse order because each layer depends on the gradient from the layer ahead of it. This is the chain rule to compute how much Layer1 contributed to the loss, we first need to know Layer2's gradient, and for Layer2 we first need Layer3's gradient. So gradients must flow backwards from loss to input, one layer at a time

In [89]:
torch.manual_seed(0)
model = Model([
    Linear(theta_1=torch.randn(8, 4), theta_0=torch.zeros(1, 8)),
    ReLU(),
    Linear(theta_1=torch.randn(1, 8), theta_0=torch.zeros(1, 1)),
])

y_pred = model.forward(X)
gout   = mse_backward(y_pred, y)
model.backward(gout)

# check gradients exist for all linear layers
for i, layer in enumerate(model.layers):
    if isinstance(layer, Linear):
        print(f'Layer {i}: theta_1.g shape={layer.theta_1.g.shape}, theta_0.g shape={layer.theta_0.g.shape}')

Layer 0: theta_1.g shape=torch.Size([8, 4]), theta_0.g shape=torch.Size([1, 8])
Layer 2: theta_1.g shape=torch.Size([1, 8]), theta_0.g shape=torch.Size([1, 1])


In [90]:
# verify all parameter gradients against PyTorch autograd
torch.manual_seed(0)
t1_0 = torch.randn(8, 4, requires_grad=True)
t0_0 = torch.zeros(1, 8, requires_grad=True)
t1_1 = torch.randn(1, 8, requires_grad=True)
t0_1 = torch.zeros(1, 1, requires_grad=True)

# autograd forward
z0   = X @ t1_0.T + t0_0
a0   = z0.clamp(0)
z1   = a0 @ t1_1.T + t0_1
loss = torch.mean((z1 - y) ** 2)
loss.backward()

# compare
l0, l1 = model.layers[0], model.layers[2]
grad_checker(l0.theta_1.g, t1_0.grad, name='layer0 theta_1')
grad_checker(l0.theta_0.g, t0_0.grad, name='layer0 theta_0')
grad_checker(l1.theta_1.g, t1_1.grad, name='layer1 theta_1')
grad_checker(l1.theta_0.g, t0_1.grad, name='layer1 theta_0')

[layer0 theta_1] max absolute error: 0.00e+00
[layer0 theta_1] max relative error: 0.00e+00
[layer0 theta_1] Gradient check PASSED.
[layer0 theta_0] max absolute error: 0.00e+00
[layer0 theta_0] max relative error: 0.00e+00
[layer0 theta_0] Gradient check PASSED.
[layer1 theta_1] max absolute error: 0.00e+00
[layer1 theta_1] max relative error: 0.00e+00
[layer1 theta_1] Gradient check PASSED.
[layer1 theta_0] max absolute error: 0.00e+00
[layer1 theta_0] max relative error: 0.00e+00
[layer1 theta_0] Gradient check PASSED.


---
## Section 5 — Deeper network

Build the following architecture and verify the full forward and backward pass:

$$\text{Linear}(4 \to 5) \to \text{ReLU} \to \text{Linear}(5 \to 10) \to \text{ReLU} \to \text{Linear}(10 \to 4) \to \text{ReLU} \to \text{Linear}(4 \to 1)$$

**Task 5.1** Define `deep_model` with the architecture above and verify the forward and backward pass.

In [91]:
torch.manual_seed(0)

# replace deep_model = None with this:
deep_model = Model([
    Linear(theta_1=torch.randn(5, 4),    # 4 inputs → 5 neurons
           theta_0=torch.zeros(1, 5)),
    ReLU(),
    Linear(theta_1=torch.randn(10, 5),   # 5 inputs → 10 neurons
           theta_0=torch.zeros(1, 10)),
    ReLU(),
    Linear(theta_1=torch.randn(4, 10),   # 10 inputs → 4 neurons
           theta_0=torch.zeros(1, 4)),
    ReLU(),
    Linear(theta_1=torch.randn(1, 4),    # 4 inputs → 1 output
           theta_0=torch.zeros(1, 1)),
])

# verify forward pass
y_pred = deep_model.forward(X)
print(f'Output shape: {y_pred.shape}')   # expect (500, 1)

# verify backward pass
deep_model.backward(mse_backward(y_pred, y))
print('Backward pass completed.')

for i, layer in enumerate(deep_model.layers):
    if isinstance(layer, Linear):
        print(f'Layer {i}: theta_1.g={layer.theta_1.g.shape}')

Output shape: torch.Size([500, 1])
Backward pass completed.
Layer 0: theta_1.g=torch.Size([5, 4])
Layer 2: theta_1.g=torch.Size([10, 5])
Layer 4: theta_1.g=torch.Size([4, 10])
Layer 6: theta_1.g=torch.Size([1, 4])


In [92]:
# verify against PyTorch autograd — rebuild same network with requires_grad
from layers import mse_forward

torch.manual_seed(0)
t1_0 = torch.randn(5,  4, requires_grad=True) 
t0_0 = torch.zeros(1,  5, requires_grad=True)
t1_1 = torch.randn(10, 5, requires_grad=True) 
t0_1 = torch.zeros(1, 10, requires_grad=True)
t1_2 = torch.randn(4, 10, requires_grad=True)
t0_2 = torch.zeros(1,  4, requires_grad=True)
t1_3 = torch.randn(1,  4, requires_grad=True) 
t0_3 = torch.zeros(1,  1, requires_grad=True)

# autograd forward — no ReLU after last linear layer
a0 = (X @ t1_0.T + t0_0).clamp(0)
a1 = (a0 @ t1_1.T + t0_1).clamp(0)
a2 = (a1 @ t1_2.T + t0_2).clamp(0)
yp =  a2 @ t1_3.T + t0_3
torch.mean((yp - y) ** 2).backward()

# compare — autograd returns None for zero gradients, treat as zeros
def to_tensor(g, shape):
    return g if g is not None else torch.zeros(shape)

linear_layers = [l for l in deep_model.layers if isinstance(l, Linear)]
for j, (layer, t1_ref, t0_ref) in enumerate(
    zip(linear_layers, [t1_0, t1_1, t1_2, t1_3], [t0_0, t0_1, t0_2, t0_3])
):
    grad_checker(layer.theta_1.g, to_tensor(t1_ref.grad, layer.theta_1.shape), name=f'layer {j} theta_1')
    grad_checker(layer.theta_0.g, to_tensor(t0_ref.grad, layer.theta_0.shape), name=f'layer {j} theta_0')

[layer 0 theta_1] max absolute error: 0.00e+00
[layer 0 theta_1] max relative error: 0.00e+00
[layer 0 theta_1] Gradient check PASSED.
[layer 0 theta_0] max absolute error: 0.00e+00
[layer 0 theta_0] max relative error: 0.00e+00
[layer 0 theta_0] Gradient check PASSED.
[layer 1 theta_1] max absolute error: 0.00e+00
[layer 1 theta_1] max relative error: 0.00e+00
[layer 1 theta_1] Gradient check PASSED.
[layer 1 theta_0] max absolute error: 2.89e-02
[layer 1 theta_0] max relative error: 1.00e+00
[layer 1 theta_0] Gradient check FAILED — check your implementation.
[layer 2 theta_1] max absolute error: 0.00e+00
[layer 2 theta_1] max relative error: 0.00e+00
[layer 2 theta_1] Gradient check PASSED.
[layer 2 theta_0] max absolute error: 1.44e-02
[layer 2 theta_0] max relative error: 5.78e-02
[layer 2 theta_0] Gradient check FAILED — check your implementation.
[layer 3 theta_1] max absolute error: 0.00e+00
[layer 3 theta_1] max relative error: 0.00e+00
[layer 3 theta_1] Gradient check PASSED.

**Question 5.1** Does the number of layers affect whether the gradient check passes?

*Answer:* No, the number of layers does not affect the gradient check. As long as each layer correctly implements its own forward and backward pass, the chain rule ensures gradients flow correctly through any number of layers. Adding more layers just adds more steps, but each step remains mathematically correct.